# Experimental Activity: Exploring Stream Processing with Apache Spark

## Learning Objectives
By the end of this activity, you will be able to:
* Explain how Spark Streaming divides event-time data into tumbling and sliding windows.
* Observe and interpret micro-batch execution and state updates in structured streaming.
* Understand the role of watermarks in handling late-arriving data.
* Reflect on trade-offs between window duration, update modes, and latency.

## Experiment Overview
In this activity, we will simulate real-time stream processing using Spark Structured Streaming. Spark will continuously monitor a designated input directory (`streaming_input/`) and automatically read any new CSV file placed there. By copying files one at a time into this directory, we will mimic live sensor data arriving in chronological order.

In [1]:
# ==========================================
# MASTER SETUP CELL: RUN THIS FIRST
# ==========================================
import os
import shutil
import time
import glob

# 1. Import ALL required PySpark SQL Types
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

# 2. Import ALL required PySpark Functions
from pyspark.sql.functions import col, to_timestamp, window, avg, min, max, count, approx_count_distinct

# 3. Import Spark Session
from pyspark.sql import SparkSession

# 4. Initialize Spark Session
spark = SparkSession.builder \
    .appName("StreamProcessingActivity") \
    .master("local[*]") \
    .getOrCreate()

# Reduce Spark console spam
spark.sparkContext.setLogLevel("WARN")

# 5. Define Kaggle Directories
SOURCE_DATA_DIR = "/kaggle/input/datasets/vsnihal/t4-ada-week5-sample-data/sample_data"
INPUT_DIR = "/kaggle/working/streaming_input"
CHECKPOINT_DIR = "/kaggle/working/checkpoints"

# Create the writable directories for Kaggle
os.makedirs(INPUT_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print("Spark Session ready. Directories created!")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/11 19:48:19 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark Session ready. Directories created!


## Base Streaming DataFrame Setup (Part 1)
Here we define the input stream for all experiments. We continuously monitor `streaming_input/` and read new CSV files as they are added. 

Setting `maxFilesPerTrigger=1` ensures only one file is processed per micro-batch. We also parse the schema and cast the timestamp string into a proper Spark `TimestampType` to act as our `event_time`.

In [3]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

# 1. Schema for the sensor data
SENSOR_SCHEMA = StructType([
    StructField("sensor_id", StringType(), True),
    StructField("location", StringType(), True),
    StructField("timestamp", StringType(), True),
    StructField("temperature", DoubleType(), True),
    StructField("humidity", DoubleType(), True)
])

# 2. Set up the base readStream
# (e.g., 'yyyy-MM-dd HH:mm:ss')
base_stream_df = spark.readStream \
    .schema(SENSOR_SCHEMA) \
    .option("maxFilesPerTrigger", 1) \
    .csv(INPUT_DIR)

# 3. Event_time column
stream_df = base_stream_df.withColumn("event_time", to_timestamp("timestamp"))

# Confirm the schema
print("Streaming DataFrame Schema:")
stream_df.printSchema()

Streaming DataFrame Schema:
root
 |-- sensor_id: string (nullable = true)
 |-- location: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- temperature: double (nullable = true)
 |-- humidity: double (nullable = true)
 |-- event_time: timestamp (nullable = true)



## Experiment 1: Tumbling Windows over Time
Observe how Spark groups incoming events into fixed-duration (non-overlapping) windows and computes average temperature, humidity, and record count for each window.

**Instructions:**
1. Run this cell.
2. Manually copy CSV files from `sample_data/` to `streaming_input/` one by one.
3. Observe the micro-batches processing in the console output.

In [4]:
WINDOW_DURATION = "30 seconds"

# TODO: Implement Tumbling Windows
exp1_df = stream_df \
    .groupBy(window(col("event_time"), WINDOW_DURATION)) \
    .agg(
        avg("temperature").alias("avg_temperature"),
        avg("humidity").alias("avg_humidity"),
        count("*").alias("record_count")
    )

# Start the stream
q1 = exp1_df.writeStream \
    .outputMode("update") \
    .format("console") \
    .option("truncate", False) \
    .queryName("exp1_tumbling") \
    .start()

# Note: The cell will continue running. You must stop it manually or use the reset cell below.

26/05/11 18:45:15 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-c709f229-309c-4cf9-80e6-4be4d75517fb. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/05/11 18:45:15 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


## 🛑 Helper: Reset Stream Environment
Before moving to the next experiment, you must stop the active streaming query and clear the checkpoint/input directories to ensure a clean state. Run this cell whenever you need to start fresh.

In [ ]:
def reset_environment(query):
    # Stop the query
    if query and query.isActive:
        query.stop()
        print(f"Query '{query.name}' stopped.")
        
    # Clear input directory
    for filename in os.listdir(INPUT_DIR):
        file_path = os.path.join(INPUT_DIR, filename)
        try:
            if os.path.isfile(file_path):
                os.unlink(file_path)
        except Exception as e:
            print(f"Failed to delete {file_path}. Reason: {e}")
            
    # Clear checkpoints
    if os.path.exists(CHECKPOINT_DIR):
        shutil.rmtree(CHECKPOINT_DIR)
        
    print("Environment reset complete. You can now start the next experiment.")

# Run reset for Experiment 1
reset_environment(q1)

## Experiment 2: Sliding Windows per Sensor with Watermarking
This builds directly on Experiment 1 by adding three new dimensions:
1. Each sensor is aggregated individually (`groupBy(sensor_id)`).
2. Windows overlap in time (`SLIDE_DURATION`).
3. A watermark controls how long Spark keeps event-time state.

**Testing Late Data:**
Try copying `sensor_data_001.csv`, `sensor_data_002.csv`, skipping 003, and adding `sensor_data_004.csv`. Then, add `sensor_data_003.csv` and observe how the watermark handles the late data!

In [ ]:
WINDOW_DURATION = "30 seconds"
SLIDE_DURATION = "10 seconds"
WATERMARK = "1 minute"

# TODO: Implement Sliding Windows with Watermarking
exp2_df = stream_df \
    .withWatermark("event_time", WATERMARK) \
    .groupBy(
        window(col("event_time"), WINDOW_DURATION, SLIDE_DURATION),
        col("sensor_id")
    ) \
    .agg(
        avg("temperature").alias("avg_temp"),
        min("temperature").alias("min_temp"),
        max("temperature").alias("max_temp"),
        count("*").alias("record_count")
    )

q2 = exp2_df.writeStream \
    .outputMode("update") \
    .format("console") \
    .option("truncate", False) \
    .option("checkpointLocation", CHECKPOINT_DIR + "/exp2") \
    .queryName("exp2_sliding_watermark") \
    .start()

## 🛑 Stop Experiment 2
Run this before proceeding to Experiment 3.

In [ ]:
reset_environment(q2)

## Experiment 3: Approximate Distinct Counting [Optional]
This shows how Spark can estimate the number of unique sensors per location using `approx_count_distinct()` with different accuracy levels within a fixed 60-second window.

In [ ]:
# TODO: Implement Approximate Distinct Counting
exp3_df = stream_df \
    .groupBy(
        window(col("event_time"), "60 seconds"),
        col("location")
    ) \
    .agg(
        # The second parameter (default ~0.05) controls the relative standard deviation (RSD)
        approx_count_distinct("sensor_id", 0.05).alias("approx_unique_sensors")
    )

q3 = exp3_df.writeStream \
    .outputMode("update") \
    .format("console") \
    .option("truncate", False) \
    .option("checkpointLocation", CHECKPOINT_DIR + "/exp3") \
    .queryName("exp3_approx_count") \
    .start()

## 🛑 Final Cleanup
Stop the final query when you are done.

In [ ]:
reset_environment(q3)

In [ ]:
import time
import glob

# Get all CSV files from the Kaggle dataset folder and sort them alphabetically
data_files = sorted(glob.glob(os.path.join(SOURCE_DATA_DIR, "*.csv")))

print(f"Found {len(data_files)} files. Starting stream simulation...\n")

for file_path in data_files:
    file_name = os.path.basename(file_path)
    target_path = os.path.join(INPUT_DIR, file_name)
    
    # Copy the file into the folder Spark is watching
    shutil.copy(file_path, target_path)
    print(f"[{time.strftime('%X')}] Added {file_name} to the stream!")
    
    # Wait 10 seconds before sending the next file to let Spark process it
    time.sleep(10)
    
print("\nAll files streamed!")

Found 100 files. Starting stream simulation...

[18:45:52] Added sensor_data_001.csv to the stream!


26/05/11 18:45:55 ERROR Executor: Exception in task 0.0 in stage 0.0 (TID 0)/ 1]
org.apache.spark.SparkDateTimeException: [CAST_INVALID_INPUT] The value 'location' of the type "STRING" cannot be cast to "TIMESTAMP" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018
== DataFrame ==
"to_timestamp" was called from
java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method)

	at org.apache.spark.sql.errors.ExecutionErrors.invalidInputInCastToDatetimeErrorInternal(ExecutionErrors.scala:115)
	at org.apache.spark.sql.errors.ExecutionErrors.invalidInputInCastToDatetimeErrorInternal$(ExecutionErrors.scala:102)
	at org.apache.spark.sql.errors.ExecutionErrors$.invalidInputInCastToDatetimeErrorInternal(ExecutionErrors.scala:259)
	at org.apache.spark.sql.errors.ExecutionErrors.invalidInputInCastToDatetimeError(ExecutionErrors.scala:92)
	at org.apache.spark.sql

[18:46:02] Added sensor_data_002.csv to the stream!
[18:46:12] Added sensor_data_003.csv to the stream!
[18:46:22] Added sensor_data_004.csv to the stream!
